# Clustering for Spatial Hotspot Detection### K-Means vs K-Means++ vs DBSCAN vs HDBSCAN**Application framing.** We treat clustering as a tool for two different jobs:| Dataset | Job | Ground truth? ||---|---|---|| D1 Earthquakes (USGS) | **Discovery** — find active fault segments / aftershock sequences | No || D2 Urban incidents (Chicago) | **Discovery** — find crime hotspots for patrol allocation | Weak proxy (district) || D3 Ecoli (UCI) | **Validation** — can unsupervised methods recover known structure? | Yes |**Central research question.** On D1/D2 no ground truth exists, so we can only use *internal* metrics(Silhouette, Davies–Bouldin, Calinski–Harabasz). These are known to reward compact convex clusters andtherefore systematically favour K-Means. D3 has labels, so we can check whether internal metrics would haveled us to the *right* method choice. If Silhouette prefers K-Means while ARI prefers DBSCAN, then anyconclusion we draw from internal metrics on the spatial data is suspect.---### How to run1. Run Section 0–1 once with internet — data is cached to `./data/`.2. Everything after that runs offline.3. Total runtime on a laptop: a few minutes.

## 0. Setup

In [ ]:
import os, time, json, warnings, urllib.parsefrom pathlib import Pathimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltfrom sklearn.cluster import KMeans, DBSCAN, HDBSCANfrom sklearn.preprocessing import StandardScalerfrom sklearn.decomposition import PCAfrom sklearn.neighbors import NearestNeighborsfrom sklearn.metrics import (    silhouette_score, davies_bouldin_score, calinski_harabasz_score,    adjusted_rand_score, normalized_mutual_info_score,)warnings.filterwarnings("ignore", category=FutureWarning)RANDOM_STATE = 42EARTH_KM     = 6371.0088          # mean Earth radius, for haversine eps conversionDATA = Path("data");    DATA.mkdir(exist_ok=True)OUT  = Path("results"); OUT.mkdir(exist_ok=True)FIG  = Path("figures"); FIG.mkdir(exist_ok=True)plt.rcParams.update({    "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",    "font.size": 9, "axes.grid": True, "grid.alpha": 0.25,})print("sklearn OK — HDBSCAN available natively, no extra install needed.")

## 1. Data loadingEach loader caches to `data/`. Re-running is free after the first download.Every loader returns `(df, X, y_true_or_None, meta)` so downstream code is uniform.

### 1.1 D1 — Earthquake epicenters (USGS FDSN Event API)

In [ ]:
USGS_URL = "https://earthquake.usgs.gov/fdsnws/event/1/query"# --- Ridgecrest 2019 sequence, Southern California -------------------------# Swap these for any region/window. Raise minmagnitude if you get too many rows.USGS_PARAMS = dict(    format="csv",    starttime="2019-07-04", endtime="2019-08-04",    minlatitude=35.0, maxlatitude=36.3,    minlongitude=-118.2, maxlongitude=-117.0,    minmagnitude=2.0,    orderby="time",)def load_earthquakes(params=USGS_PARAMS, cache="data/earthquakes.csv", force=False):    p = Path(cache)    if force or not p.exists():        url = USGS_URL + "?" + urllib.parse.urlencode(params)        print("downloading:", url)        pd.read_csv(url).to_csv(p, index=False)    df = pd.read_csv(p)    df = df.dropna(subset=["latitude", "longitude"]).reset_index(drop=True)    X  = df[["latitude", "longitude"]].to_numpy()    meta = dict(name="Earthquakes (Ridgecrest 2019)", spatial=True,                units="degrees lat/lon", job="discovery")    return df, X, None, metaeq_df, eq_X, eq_y, eq_meta = load_earthquakes()print(eq_df.shape, "events")eq_df[["time", "latitude", "longitude", "depth", "mag"]].head()

### 1.2 D2 — Urban incidents (Chicago open data, Socrata API)

In [ ]:
CHI_URL = "https://data.cityofchicago.org/resource/ijzp-q8t2.csv"CHI_PARAMS = {    "$where": "date between '2023-07-01T00:00:00' and '2023-07-31T23:59:59'",    "primary_type": "THEFT",    "$select": "id,date,primary_type,latitude,longitude,district,community_area,beat",    "$limit": 6000,}def load_chicago(params=CHI_PARAMS, cache="data/chicago.csv", force=False):    p = Path(cache)    if force or not p.exists():        url = CHI_URL + "?" + urllib.parse.urlencode(params)        print("downloading:", url)        pd.read_csv(url).to_csv(p, index=False)    df = pd.read_csv(p)    df = df.dropna(subset=["latitude", "longitude", "district"]).reset_index(drop=True)    # Block-level rounding creates exact duplicate coordinates. Jitter by ~5 m so    # duplicated points do not artificially inflate local density for DBSCAN.    rng = np.random.default_rng(RANDOM_STATE)    jit = rng.normal(0, 5e-5, size=(len(df), 2))    X = df[["latitude", "longitude"]].to_numpy() + jit    y = df["district"].astype(int).to_numpy()   # weak proxy label — see caveat below    meta = dict(name="Chicago THEFT (Jul 2023)", spatial=True,                units="degrees lat/lon", job="discovery")    return df, X, y, metachi_df, chi_X, chi_y, chi_meta = load_chicago()print(chi_df.shape, "incidents |", chi_df["district"].nunique(), "police districts")chi_df.head()

> **Caveat to state explicitly in the report.** `district` is an *administrative* boundary, not a natural> cluster. A high ARI against it would mean the algorithm recovered policing geography, not crime structure.> We report it as a weak reference only and never treat it as truth.

### 1.3 D3 — Ecoli protein localization (UCI) — the labelled validation set

In [ ]:
ECOLI_COLS = ["seq_name","mcg","gvh","lip","chg","aac","alm1","alm2","site"]ECOLI_URLS = [    "https://archive.ics.uci.edu/ml/machine-learning-databases/ecoli/ecoli.data",    "https://archive.ics.uci.edu/static/public/39/data.csv",]def load_ecoli(cache="data/ecoli.csv", force=False):    p = Path(cache)    if force or not p.exists():        last = None        for url in ECOLI_URLS:            try:                print("trying:", url)                d = pd.read_csv(url, sep=r"\s+", header=None, names=ECOLI_COLS)                d.to_csv(p, index=False); break            except Exception as e:                last = e        else:            raise RuntimeError(f"Could not fetch Ecoli. Download manually to {p}. Last error: {last}")    df = pd.read_csv(p)    feats = ["mcg","gvh","lip","chg","aac","alm1","alm2"]    X = df[feats].to_numpy(float)    y = pd.Categorical(df["site"]).codes    meta = dict(name="Ecoli (protein localization)", spatial=False,                units="normalized scores", job="validation")    return df, X, y, metaec_df, ec_X, ec_y, ec_meta = load_ecoli()print(ec_df.shape)print(ec_df["site"].value_counts())

Note the class imbalance: several localization sites have only 2–5 members. This matters —DBSCAN will almost certainly label them noise, and whether that is a *failure* or *correct behaviour*is a real question we answer in Section 8.

### 1.4 Optional — synthetic control (one line, no download)

In [ ]:
from sklearn.datasets import make_moonssyn_X, syn_y = make_moons(n_samples=1000, noise=0.08, random_state=RANDOM_STATE)syn_meta = dict(name="make_moons (control)", spatial=False, units="arbitrary", job="mechanism")print(syn_X.shape)

### 1.5 Dataset registry

In [ ]:
DATASETS = {    "earthquakes": dict(X=eq_X,  y=eq_y,  meta=eq_meta,  df=eq_df),    "chicago":     dict(X=chi_X, y=chi_y, meta=chi_meta, df=chi_df),    "ecoli":       dict(X=ec_X,  y=ec_y,  meta=ec_meta,  df=ec_df),    "moons":       dict(X=syn_X, y=syn_y, meta=syn_meta, df=None),}summary = pd.DataFrame([    {"dataset": k, "n": len(v["X"]), "dims": v["X"].shape[1],     "classes": (len(np.unique(v["y"])) if v["y"] is not None else None),     "spatial": v["meta"]["spatial"], "job": v["meta"]["job"]}    for k, v in DATASETS.items()])summary

## 2. PreprocessingTwo decisions that materially change results and deserve a paragraph each in the report.

### 2.1 Distance metric for geographic coordinatesTreating lat/lon as Euclidean distorts distance away from the equator — at Chicago's latitude one degree oflongitude is ~30% shorter than one of latitude. Two valid fixes:* **Haversine** metric with coordinates in radians (`eps` in radians = km / 6371)* **Projection** to a local metric CRS so `eps` is directly in metresWe use haversine, which also makes `eps` *interpretable*: "a hotspot is a neighbourhood of events within300 m". K-Means has no comparable interpretable parameter — `k` is a guess. That is a real practicalargument for DBSCAN in this application, independent of any metric score.

In [ ]:
def to_radians(coords):    return np.radians(np.asarray(coords, float))def km_to_rad(km):    return km / EARTH_KMdef rad_to_km(rad):    return rad * EARTH_KMdef preprocess(name, scale=True):    d = DATASETS[name]    X, meta = d["X"], d["meta"]    if meta["spatial"]:        # spatial: keep raw degrees for plotting, radians for haversine fitting        return dict(X_fit=to_radians(X), X_plot=X, metric="haversine", spatial=True)    Xs = StandardScaler().fit_transform(X) if scale else X.astype(float)    Xp = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(Xs) if Xs.shape[1] > 2 else Xs    return dict(X_fit=Xs, X_plot=Xp, metric="euclidean", spatial=False)PREP = {k: preprocess(k) for k in DATASETS}{k: (v["X_fit"].shape, v["metric"]) for k, v in PREP.items()}

### 2.2 Visualise the raw data

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.6))for ax, (k, p) in zip(axes, PREP.items()):    Xp = p["X_plot"]    ax.scatter(Xp[:, 1] if p["spatial"] else Xp[:, 0],               Xp[:, 0] if p["spatial"] else Xp[:, 1],               s=4, alpha=0.45, c="k", linewidths=0)    ax.set_title(f"{DATASETS[k]['meta']['name']}\n(n={len(Xp)})", fontsize=9)    ax.set_xlabel("longitude" if p["spatial"] else "dim 1")    ax.set_ylabel("latitude"  if p["spatial"] else "dim 2")plt.tight_layout(); plt.savefig(FIG/"fig1_datasets.png"); plt.show()

## 3. Evaluation harnessOne function, used everywhere. Writing this before running any experiment is what keeps four people'sresults comparable.**Noise handling rule:** internal metrics are computed on non-noise points only (they are undefined for a"noise" pseudo-cluster), while ARI/NMI use all points with noise as its own label. State this in the paper —it is a real methodological choice that affects the numbers.

In [ ]:
def evaluate(X, labels, y_true=None, exclude_noise_internal=True):    lab = np.asarray(labels)    res = {        "n_clusters":  int(len(set(lab[lab != -1]))),        "n_noise":     int((lab == -1).sum()),        "noise_frac":  float((lab == -1).mean()),    }    m = (lab != -1) if exclude_noise_internal else np.ones(len(lab), bool)    ok = res["n_clusters"] >= 2 and m.sum() > res["n_clusters"]    res["silhouette"]        = float(silhouette_score(X[m], lab[m]))        if ok else np.nan    res["davies_bouldin"]    = float(davies_bouldin_score(X[m], lab[m]))    if ok else np.nan    res["calinski_harabasz"] = float(calinski_harabasz_score(X[m], lab[m])) if ok else np.nan    if y_true is not None:        res["ari"] = float(adjusted_rand_score(y_true, lab))        res["nmi"] = float(normalized_mutual_info_score(y_true, lab))    else:        res["ari"] = res["nmi"] = np.nan    return resdef run(estimator, X, y_true=None, **extra):    t0 = time.perf_counter()    lab = estimator.fit_predict(X)    dt  = time.perf_counter() - t0    row = {"runtime_s": round(dt, 4), **extra}    row.update(evaluate(X, lab, y_true))    return row, labprint("harness ready")

## 4. Parameter selectionWe do **not** hand-tune. Each algorithm gets a documented, reproducible selection procedure — this is whatturns "we picked eps=0.3" into a defensible method section.

### 4.1 k for K-Means — elbow + silhouette

In [ ]:
def k_sweep(X, ks=range(2, 21), metric="euclidean", y_true=None):    rows = []    for k in ks:        km = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=RANDOM_STATE)        lab = km.fit_predict(X)        r = {"k": k, "inertia": km.inertia_}        r.update(evaluate(X, lab, y_true))        rows.append(r)    return pd.DataFrame(rows)k_results = {}for name in DATASETS:    p = PREP[name]    k_results[name] = k_sweep(p["X_fit"], y_true=DATASETS[name]["y"])fig, axes = plt.subplots(2, 4, figsize=(16, 6), sharex=True)for j, name in enumerate(DATASETS):    d = k_results[name]    axes[0, j].plot(d["k"], d["inertia"], "o-", ms=3)    axes[0, j].set_title(DATASETS[name]["meta"]["name"], fontsize=9)    axes[0, j].set_ylabel("inertia (elbow)")    axes[1, j].plot(d["k"], d["silhouette"], "o-", ms=3, label="silhouette")    if d["ari"].notna().any():        axes[1, j].plot(d["k"], d["ari"], "s-", ms=3, label="ARI")    axes[1, j].set_xlabel("k"); axes[1, j].legend(fontsize=7)axes[1, 0].set_ylabel("score")plt.tight_layout(); plt.savefig(FIG/"fig2_k_selection.png"); plt.show()BEST_K = {n: int(d.loc[d["silhouette"].idxmax(), "k"]) for n, d in k_results.items()}print("k chosen by silhouette:", BEST_K)

### 4.2 eps for DBSCAN — k-distance curve with automatic knee detectionThe standard heuristic: sort each point's distance to its `min_samples`-th neighbour and take the knee.We detect the knee as the point of maximum perpendicular distance from the chord joining the curve'sendpoints — deterministic and reportable, unlike eyeballing.

In [ ]:
def k_distance(X, min_samples=5, metric="euclidean"):    algo = "ball_tree" if metric == "haversine" else "auto"    nn = NearestNeighbors(n_neighbors=min_samples, metric=metric, algorithm=algo).fit(X)    d, _ = nn.kneighbors(X)    return np.sort(d[:, -1])def knee(curve):    x = np.arange(len(curve), dtype=float); y = np.asarray(curve, float)    p1, p2 = np.array([x[0], y[0]]), np.array([x[-1], y[-1]])    v = (p2 - p1) / np.linalg.norm(p2 - p1)    pts = np.c_[x, y] - p1    dist = np.abs(pts[:, 0]*v[1] - pts[:, 1]*v[0])    i = int(dist.argmax())    return i, float(y[i])EPS = {}fig, axes = plt.subplots(1, 4, figsize=(16, 3.2))for ax, name in zip(axes, DATASETS):    p = PREP[name]; ms = 5    kd = k_distance(p["X_fit"], min_samples=ms, metric=p["metric"])    i, e = knee(kd)    EPS[name] = e    disp = rad_to_km(kd) if p["spatial"] else kd    ax.plot(disp); ax.axvline(i, color="r", ls="--", lw=1)    lbl = f"eps = {rad_to_km(e):.2f} km" if p["spatial"] else f"eps = {e:.3f}"    ax.axhline(disp[i], color="r", ls=":", lw=1)    ax.set_title(f"{name}\n{lbl}", fontsize=9)    ax.set_xlabel("points sorted"); ax.set_ylabel("km" if p["spatial"] else "distance")plt.tight_layout(); plt.savefig(FIG/"fig3_kdistance.png"); plt.show()print({n: (round(rad_to_km(e),3) if PREP[n]["spatial"] else round(e,4)) for n, e in EPS.items()})

### 4.3 Guard: does the knee eps actually produce clusters?The knee heuristic can fail — on data where density is roughly uniform (which is common inmoderate-dimensional feature spaces like Ecoli) it returns an `eps` so large that DBSCAN mergeseverything into one cluster. **This is a finding, not a bug**, and you should report it: it is directevidence that the standard DBSCAN heuristic degrades outside low-dimensional spatial data.Below we keep the knee value, detect the degenerate case, and record a fallback so the comparisoncan still proceed. Report both numbers in the paper.

In [ ]:
EPS_FALLBACK, EPS_NOTES = {}, {}def refine_eps(name, min_samples=5, min_clusters=2):    """Keep the knee eps if it yields >=min_clusters; else search downward and record that it failed."""    p = PREP[name]    algo = "ball_tree" if p["metric"] == "haversine" else "auto"    base = EPS[name]    def n_clus(e):        lab = DBSCAN(eps=e, min_samples=min_samples, metric=p["metric"],                     algorithm=algo).fit_predict(p["X_fit"])        return len(set(lab[lab != -1]))    if n_clus(base) >= min_clusters:        EPS_NOTES[name] = "knee eps valid"        return base    kd = k_distance(p["X_fit"], min_samples, p["metric"])    for q in [90, 80, 70, 60, 50, 40, 30, 20, 10, 5]:        e = float(np.percentile(kd, q))        if n_clus(e) >= min_clusters:            EPS_NOTES[name] = f"knee eps={base:.4g} degenerate (1 cluster); fell back to {q}th pct"            print(f"[{name}] {EPS_NOTES[name]}")            return e    EPS_NOTES[name] = "no eps in search range produced >=2 clusters"    print(f"[{name}] {EPS_NOTES[name]}")    return basefor name in DATASETS:    EPS_FALLBACK[name] = refine_eps(name)EPS = EPS_FALLBACK   # used by all downstream sectionspd.DataFrame([{"dataset": n,               "eps_used": (round(rad_to_km(EPS[n]), 4) if PREP[n]["spatial"] else round(EPS[n], 5)),               "units": "km" if PREP[n]["spatial"] else "standardized",               "note": EPS_NOTES[n]} for n in DATASETS])

## 5. Main experiment — all methods × all datasetsWrites `results/main_results.csv`. This file is the single source of truth for every number in the report.

In [ ]:
def build_methods(name, k, eps, min_samples=5):    metric = PREP[name]["metric"]    algo   = "ball_tree" if metric == "haversine" else "auto"    return {        "KMeans":    (KMeans(k, init="random",    n_init=10, random_state=RANDOM_STATE),                      dict(params=f"k={k}, init=random")),        "KMeans++":  (KMeans(k, init="k-means++", n_init=10, random_state=RANDOM_STATE),                      dict(params=f"k={k}, init=k-means++")),        "DBSCAN":    (DBSCAN(eps=eps, min_samples=min_samples, metric=metric, algorithm=algo),                      dict(params=f"eps={eps:.5f}, min_samples={min_samples}")),        "HDBSCAN":   (HDBSCAN(min_cluster_size=max(5, int(0.01*len(PREP[name]['X_fit']))),                              metric=metric if metric != "haversine" else "haversine",                              copy=True),                      dict(params="min_cluster_size=1% of n")),    }rows, LABELS = [], {}for name in DATASETS:    p, y = PREP[name], DATASETS[name]["y"]    for mname, (est, info) in build_methods(name, BEST_K[name], EPS[name]).items():        r, lab = run(est, p["X_fit"], y, dataset=name, method=mname, **info)        rows.append(r); LABELS[(name, mname)] = labmain = pd.DataFrame(rows)[    ["dataset","method","params","n_clusters","noise_frac","silhouette",     "davies_bouldin","calinski_harabasz","ari","nmi","runtime_s"]]main.to_csv(OUT/"main_results.csv", index=False)main.round(4)

## 6. Qualitative results — where each method visibly succeeds or failsThe most persuasive figure in the paper. A K-Means hotspot drawn over an empty area needs no metric tomake the point.

In [ ]:
def plot_grid(dataset_names=("earthquakes","chicago","ecoli","moons"),              methods=("KMeans","KMeans++","DBSCAN","HDBSCAN")):    fig, axes = plt.subplots(len(dataset_names), len(methods)+1,                             figsize=(3.0*(len(methods)+1), 3.0*len(dataset_names)))    for i, dn in enumerate(dataset_names):        p, y = PREP[dn], DATASETS[dn]["y"]        Xp = p["X_plot"]        xs, ys = (Xp[:,1], Xp[:,0]) if p["spatial"] else (Xp[:,0], Xp[:,1])        ax = axes[i,0]        if y is not None:            ax.scatter(xs, ys, c=y, s=4, cmap="tab20", alpha=.7, linewidths=0)            ax.set_title("ground truth / reference", fontsize=8)        else:            ax.scatter(xs, ys, c="k", s=4, alpha=.4, linewidths=0)            ax.set_title("raw (no ground truth)", fontsize=8)        ax.set_ylabel(DATASETS[dn]["meta"]["name"], fontsize=8)        for j, mn in enumerate(methods, start=1):            lab = LABELS[(dn, mn)]            ax = axes[i,j]            noise = lab == -1            ax.scatter(xs[~noise], ys[~noise], c=lab[~noise], s=4, cmap="tab20", alpha=.75, linewidths=0)            if noise.any():                ax.scatter(xs[noise], ys[noise], c="lightgray", s=3, alpha=.5, linewidths=0)            r = main[(main.dataset==dn)&(main.method==mn)].iloc[0]            ax.set_title(f"{mn}\nclus={r.n_clusters}, noise={r.noise_frac:.0%}", fontsize=8)    for ax in axes.ravel(): ax.set_xticks([]); ax.set_yticks([])    plt.tight_layout(); plt.savefig(FIG/"fig4_qualitative_grid.png"); plt.show()plot_grid()

## 7. Ablation studiesFour ablations, each isolating one component.

### 7.1 Ablation A — distance metric (haversine vs Euclidean-on-degrees)

In [ ]:
ab_metric = []for name in ["earthquakes", "chicago"]:    deg = DATASETS[name]["X"]; rad = to_radians(deg)    eps_r = EPS[name]; eps_d = np.degrees(eps_r)   # naive equivalent in degrees    for label, Xf, mtr, e in [("haversine", rad, "haversine", eps_r),                              ("euclidean_deg", deg, "euclidean", eps_d)]:        algo = "ball_tree" if mtr == "haversine" else "auto"        r, _ = run(DBSCAN(eps=e, min_samples=5, metric=mtr, algorithm=algo),                   Xf, DATASETS[name]["y"], dataset=name, metric_used=label)        ab_metric.append(r)ab_metric = pd.DataFrame(ab_metric)[["dataset","metric_used","n_clusters","noise_frac","silhouette","ari"]]ab_metric.to_csv(OUT/"ablation_metric.csv", index=False)ab_metric.round(4)

### 7.2 Ablation B — feature scaling (Ecoli)

In [ ]:
ab_scale = []for label, Xf in [("raw", DATASETS["ecoli"]["X"].astype(float)),                  ("standardized", StandardScaler().fit_transform(DATASETS["ecoli"]["X"]))]:    k = BEST_K["ecoli"]    kd = k_distance(Xf, 5); _, e = knee(kd)    for mn, est in [("KMeans++", KMeans(k, n_init=10, random_state=RANDOM_STATE)),                    ("DBSCAN",   DBSCAN(eps=e, min_samples=5)),                    ("HDBSCAN",  HDBSCAN(min_cluster_size=5, copy=True))]:        r, _ = run(est, Xf, DATASETS["ecoli"]["y"], scaling=label, method=mn)        ab_scale.append(r)ab_scale = pd.DataFrame(ab_scale)[["scaling","method","n_clusters","noise_frac","silhouette","ari","nmi"]]ab_scale.to_csv(OUT/"ablation_scaling.csv", index=False)ab_scale.round(4)

### 7.3 Ablation C — dimensionality (does DBSCAN degrade as dims grow?)

In [ ]:
Xs_ec = StandardScaler().fit_transform(DATASETS["ecoli"]["X"])ab_dim = []for d in [2, 3, 5, 7]:    Xd = PCA(n_components=d, random_state=RANDOM_STATE).fit_transform(Xs_ec)    kd = k_distance(Xd, 5); _, e = knee(kd)    for mn, est in [("KMeans++", KMeans(BEST_K["ecoli"], n_init=10, random_state=RANDOM_STATE)),                    ("DBSCAN",   DBSCAN(eps=e, min_samples=5)),                    ("HDBSCAN",  HDBSCAN(min_cluster_size=5, copy=True))]:        r, _ = run(est, Xd, DATASETS["ecoli"]["y"], dims=d, method=mn, eps_knee=round(e,4))        ab_dim.append(r)ab_dim = pd.DataFrame(ab_dim)[["dims","method","eps_knee","n_clusters","noise_frac","silhouette","ari"]]ab_dim.to_csv(OUT/"ablation_dims.csv", index=False)piv = ab_dim.pivot(index="dims", columns="method", values="ari")piv.plot(marker="o", figsize=(5,3.2)); plt.ylabel("ARI"); plt.title("Ecoli: ARI vs dimensionality")plt.tight_layout(); plt.savefig(FIG/"fig5_dimensionality.png"); plt.show()ab_dim.round(4)

### 7.4 Ablation D — initialization: K-Means vs K-Means++ over 30 seedsThe honest expectation: K-Means++ mainly reduces *variance* and iteration count rather than improving thebest-case result. Report mean ± std — a single-seed number is the most common way these projects lose marks.

In [ ]:
ab_init = []for name in DATASETS:    Xf, y, k = PREP[name]["X_fit"], DATASETS[name]["y"], BEST_K[name]    for init in ["random", "k-means++"]:        for seed in range(30):            km = KMeans(k, init=init, n_init=1, random_state=seed)            lab = km.fit_predict(Xf)            ab_init.append(dict(dataset=name, init=init, seed=seed,                                inertia=km.inertia_, n_iter=km.n_iter_,                                ari=(adjusted_rand_score(y, lab) if y is not None else np.nan),                                silhouette=silhouette_score(Xf, lab)))ab_init = pd.DataFrame(ab_init)ab_init.to_csv(OUT/"ablation_init.csv", index=False)init_summary = (ab_init.groupby(["dataset","init"])                .agg(inertia_mean=("inertia","mean"), inertia_std=("inertia","std"),                     iters_mean=("n_iter","mean"),                     sil_mean=("silhouette","mean"), sil_std=("silhouette","std"),                     ari_mean=("ari","mean"), ari_std=("ari","std"))                .reset_index())init_summary.to_csv(OUT/"ablation_init_summary.csv", index=False)init_summary.round(4)

### 7.5 DBSCAN parameter sensitivity heatmap — how narrow is the good region?

In [ ]:
def eps_grid(name, n=12, lo=0.4, hi=2.0):    base = EPS[name]    return np.linspace(base*lo, base*hi, n)MS_GRID = [3, 5, 10, 20]sens_all = []for name in DATASETS:    p, y = PREP[name], DATASETS[name]["y"]    algo = "ball_tree" if p["metric"] == "haversine" else "auto"    for e in eps_grid(name):        for ms in MS_GRID:            lab = DBSCAN(eps=e, min_samples=ms, metric=p["metric"], algorithm=algo).fit_predict(p["X_fit"])            r = evaluate(p["X_fit"], lab, y)            sens_all.append(dict(dataset=name, eps=e, min_samples=ms, **r))sens = pd.DataFrame(sens_all)sens.to_csv(OUT/"dbscan_sensitivity.csv", index=False)fig, axes = plt.subplots(1, 4, figsize=(17, 3.4))for ax, name in zip(axes, DATASETS):    s = sens[sens.dataset == name]    val = "ari" if s["ari"].notna().any() else "silhouette"    grid = s.pivot(index="min_samples", columns="eps", values=val)    im = ax.imshow(grid.values, aspect="auto", origin="lower", cmap="viridis")    ax.set_yticks(range(len(grid.index)), grid.index)    xt = np.linspace(0, len(grid.columns)-1, 4).astype(int)    scale = rad_to_km if PREP[name]["spatial"] else (lambda v: v)    ax.set_xticks(xt, [f"{scale(grid.columns[i]):.2f}" for i in xt], fontsize=7)    ax.set_xlabel("eps (km)" if PREP[name]["spatial"] else "eps")    ax.set_ylabel("min_samples"); ax.set_title(f"{name} — {val}", fontsize=9)    plt.colorbar(im, ax=ax, fraction=.046)plt.tight_layout(); plt.savefig(FIG/"fig6_dbscan_sensitivity.png"); plt.show()

## 8. Noise analysis — is DBSCAN's "noise" a bug or a feature?The same mechanism, two domains, opposite verdicts:* **Spatial (D1/D2):** isolated events genuinely are background. Labelling them noise is *correct* and is  something K-Means cannot do at all — it forces every point into a hotspot.* **Ecoli (D3):** the rare localization sites have 2–5 members. Labelling them noise *discards the signal a  biologist most cares about* — rare classes are often the interesting ones.

In [ ]:
noise_rows = []for name in DATASETS:    y = DATASETS[name]["y"]    for mn in ["DBSCAN", "HDBSCAN"]:        lab = LABELS[(name, mn)]        noise = lab == -1        row = dict(dataset=name, method=mn, noise_frac=noise.mean(), n_noise=int(noise.sum()))        if y is not None and noise.any():            # which true classes are disproportionately labelled noise?            rate = pd.Series(y[noise]).value_counts() / pd.Series(y).value_counts()            row["worst_class"] = int(rate.idxmax()); row["worst_class_noise_rate"] = float(rate.max())            sizes = pd.Series(y).value_counts()            row["worst_class_size"] = int(sizes[rate.idxmax()])        noise_rows.append(row)noise_df = pd.DataFrame(noise_rows)noise_df.to_csv(OUT/"noise_analysis.csv", index=False)noise_df.round(4)

In [ ]:
# Ecoli: noise rate as a function of true class size — the rare-class question, quantifiedy = DATASETS["ecoli"]["y"]lab = LABELS[("ecoli","DBSCAN")]tab = (pd.DataFrame({"true": y, "noise": lab == -1})       .groupby("true").agg(size=("noise","size"), noise_rate=("noise","mean")).reset_index())tab["site"] = pd.Categorical(ec_df["site"]).categories[tab["true"]]plt.figure(figsize=(5,3.2))plt.scatter(tab["size"], tab["noise_rate"], s=40)for _, r in tab.iterrows():    plt.annotate(r["site"], (r["size"], r["noise_rate"]), fontsize=7,                 xytext=(3,3), textcoords="offset points")plt.xscale("log"); plt.xlabel("true class size (log)"); plt.ylabel("fraction labelled noise")plt.title("Ecoli: DBSCAN discards small classes")plt.tight_layout(); plt.savefig(FIG/"fig7_noise_vs_classsize.png"); plt.show()tab.round(3)

## 9. Core analysis — do internal metrics pick the right method?This is the section that justifies including a labelled dataset in a spatial-application paper.Procedure: on the datasets with labels, rank the methods by each **internal** metric, then by **ARI**.If the rankings disagree, then the internal metrics we are forced to rely on for D1/D2 are unreliableguides — and every conclusion drawn from them needs a caveat.

In [ ]:
labelled = [n for n in DATASETS if DATASETS[n]["y"] is not None]def ranking(df, col, higher_is_better=True):    s = df.set_index("method")[col]    return s.sort_values(ascending=not higher_is_better).index.tolist()cmp_rows = []for name in labelled:    sub = main[main.dataset == name]    cmp_rows.append(dict(        dataset=name,        by_silhouette=" > ".join(ranking(sub, "silhouette")),        by_davies_bouldin=" > ".join(ranking(sub, "davies_bouldin", False)),        by_calinski=" > ".join(ranking(sub, "calinski_harabasz")),        by_ari=" > ".join(ranking(sub, "ari")),        agrees=ranking(sub, "silhouette")[0] == ranking(sub, "ari")[0],    ))agreement = pd.DataFrame(cmp_rows)agreement.to_csv(OUT/"metric_agreement.csv", index=False)for _, r in agreement.iterrows():    print(f"\n[{r.dataset}]")    print("  silhouette says:", r.by_silhouette)    print("  ARI        says:", r.by_ari)    print("  top-1 agree:", r.agrees)

In [ ]:
# Quantify the bias: correlation between each internal metric and ARI across ALL runs we havepool = pd.concat([    main[main.dataset.isin(labelled)][["silhouette","davies_bouldin","calinski_harabasz","ari"]],    sens[sens.dataset.isin(labelled)][["silhouette","davies_bouldin","calinski_harabasz","ari"]],]).dropna()corr = pool.corr(method="spearman")["ari"].drop("ari")print("Spearman correlation with ARI (n=%d runs):" % len(pool))print(corr.round(3))print("\nDavies-Bouldin is lower-is-better, so a NEGATIVE correlation here is the 'good' direction.")

## 10. Export tables for the reportProduces LaTeX-ready tables in `results/`. Paste with `\input{}` rather than retyping numbers —retyping is how the paper ends up disagreeing with the code.

In [ ]:
def to_latex(df, path, caption, label, float_fmt="%.3f"):    tex = df.to_latex(index=False, float_format=float_fmt, escape=True,                      caption=caption, label=label, position="t")    Path(path).write_text(tex)    print("wrote", path)to_latex(summary, OUT/"tab1_datasets.tex",         "Datasets used in this study.", "tab:datasets")to_latex(main.round(3), OUT/"tab2_main.tex",         "Main comparison across all datasets and methods.", "tab:main")to_latex(init_summary.round(3), OUT/"tab3_init.tex",         "Initialization ablation over 30 seeds (mean and standard deviation).", "tab:init")to_latex(ab_scale.round(3), OUT/"tab4_scaling.tex",         "Feature-scaling ablation on Ecoli.", "tab:scaling")to_latex(ab_dim.round(3), OUT/"tab5_dims.tex",         "Dimensionality ablation on Ecoli.", "tab:dims")print("\nAll artefacts:")for p in sorted(list(OUT.glob('*')) + list(FIG.glob('*'))):    print(" ", p)

## 11. Reproducibility recordPaste this into the paper's reproducibility paragraph.

In [ ]:
import sys, sklearn, platformrecord = {    "python": sys.version.split()[0],    "platform": platform.platform(),    "numpy": np.__version__, "pandas": pd.__version__, "sklearn": sklearn.__version__,    "random_state": RANDOM_STATE,    "n_seeds_init_ablation": 30,    "datasets": {k: dict(n=int(len(v["X"])), dims=int(v["X"].shape[1])) for k, v in DATASETS.items()},    "chosen_k": BEST_K,    "chosen_eps": {k: (round(rad_to_km(v),4) if PREP[k]["spatial"] else round(v,5)) for k,v in EPS.items()},}Path(OUT/"repro.json").write_text(json.dumps(record, indent=2))print(json.dumps(record, indent=2))

---## What to do next**Division of work across four people** — each owns a section end-to-end:| Person | Sections | Deliverable ||---|---|---|| P1 Data | 1–2 | all three datasets cached, metric choice justified || P2 Centroid | 4.1, 7.4 | k selection + init ablation with mean±std || P3 Density | 4.2, 7.5, 8 | eps heuristic, sensitivity heatmap, noise analysis || P4 Eval/Report | 3, 5–6, 9–10 | harness, figures, LaTeX tables, integration |**Before you write a word of the paper**, run everything once and look at `results/metric_agreement.csv`.If silhouette and ARI disagree on which method wins, that disagreement is your paper's main finding, andSection 9 becomes the centrepiece rather than an appendix.**Known extension points** if you want more depth:- Add OPTICS as a fifth method (`sklearn.cluster.OPTICS`) — the missing link between DBSCAN and HDBSCAN- Cluster earthquakes on (lat, lon, depth) instead of (lat, lon) and see whether 3-D structure changes fault segmentation- Add a temporal dimension to the crime data — hotspots move, and single-snapshot clustering hides that